# Sampling Assignment
**Name:** Keshav Peshawaria  
**Roll No:** 102303502

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import RandomOverSampler

In [ ]:
data = pd.read_csv('Creditcard_data.csv')
X = data.drop('Class', axis=1)
y = data['Class']

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
df_new = pd.concat([X_resampled, y_resampled], axis=1)
print(df_new['Class'].value_counts())

In [ ]:
Z = 1.96
p = 0.5
E = 0.05
n = int((Z**2 * p * (1-p)) / (E**2))
print(n)

In [ ]:
samples = {}

samples['Simple Random'] = df_new.sample(n=n, random_state=42)

k = int(len(df_new) / n)
samples['Systematic'] = df_new.iloc[::k]

samples['Stratified'] = df_new.groupby('Class', group_keys=False).apply(lambda x: x.sample(int(n/2), random_state=42))

def cluster_sampling(df, n_clusters, samples_per_cluster):
    df_clustered = df.copy()
    df_clustered['cluster_id'] = np.random.randint(0, n_clusters, size=len(df))
    selected_clusters = np.random.choice(range(n_clusters), size=samples_per_cluster, replace=False)
    return df_clustered[df_clustered['cluster_id'].isin(selected_clusters)].drop('cluster_id', axis=1)

samples['Cluster'] = cluster_sampling(df_new, 20, 5)

samples['Bootstrap'] = df_new.sample(n=n, replace=True, random_state=42)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {model_name: {} for model_name in models}

for sample_name, sample_df in samples.items():
    X_sample = sample_df.drop('Class', axis=1)
    y_sample = sample_df['Class']
    
    X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.3, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    for model_name, model in models.items():
        if model_name == 'SVM' or model_name == 'Logistic Regression':
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_test_scaled)
        else:
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
        
        acc = accuracy_score(y_test, pred)
        results[model_name][sample_name] = acc

In [ ]:
print(pd.DataFrame(results))